# MCP Server Basics — Model Context Protocol

## What is MCP?

The **Model Context Protocol (MCP)** is a standardized, open protocol that defines how LLMs
connect to external tools, data sources, and services. Think of it as a "USB-C port" for AI:
instead of every LLM needing custom integrations for every tool, MCP provides one standard
interface that any LLM client can use to talk to any MCP server.

## What You Will Learn

In this notebook you will explore:

1. **How MCP servers are built** using the `FastMCP` high-level API
2. **Three server archetypes** — filesystem, API wrapper, and database
3. **The tool-definition pattern** — decorators, docstrings, and type hints
4. **Security considerations** — path traversal protection, read-only queries

## Prerequisites

- Basic Python knowledge (functions, decorators, type hints)
- Familiarity with the concept of APIs

## Learning Resources

| Resource | Link |
|----------|------|
| MCP Specification | https://spec.modelcontextprotocol.io/ |
| Python MCP SDK | https://github.com/modelcontextprotocol/python-sdk |
| FastMCP Guide | https://github.com/modelcontextprotocol/python-sdk#fastmcp |
| Video: Build MCP Servers | https://www.youtube.com/watch?v=23mkvV2RLWU |

In [ ]:
# ── Imports: the three MCP server modules in AgentExplorr ──────────────────
# Each module creates a FastMCP server instance and decorates plain Python
# functions to turn them into tools an LLM can discover and invoke.

# The high-level SDK class used by every server
from mcp.server.fastmcp import FastMCP

# Filesystem server  — exposes read_file, write_file, list_directory, search_files
from agentexplorr.mcp.servers import filesystem_server

# API server         — wraps the free Open-Meteo weather REST API
from agentexplorr.mcp.servers import api_server

# Database server    — exposes list_tables, describe_table, run_query (read-only SQL)
from agentexplorr.mcp.servers import database_server

print("Server modules imported successfully.")
print()
print(f"Filesystem server name : {filesystem_server.server.name}")
print(f"API server name        : {api_server.server.name}")
print(f"Database server name   : {database_server.server.name}")
print()
print("Each server is a FastMCP instance that registers tools via @server.tool() decorators.")

## The Filesystem Server

The **filesystem server** (`filesystem_server.py`) is the simplest example of an MCP server.
It exposes four tools that let an LLM interact with the local file system:

| Tool | Purpose |
|------|---------|
| `read_file` | Read contents of a file |
| `write_file` | Create or overwrite a file |
| `list_directory` | List entries in a directory |
| `search_files` | Glob-search for files matching a pattern |

### How it works

1. A `FastMCP` instance is created with a descriptive `name` and `instructions` string.
2. A **base directory** (`BASE_DIR`) restricts all file operations to a safe scope.
3. A helper function `_resolve_safe_path()` blocks path-traversal attacks (e.g. `../../etc/passwd`).
4. Each tool is a plain Python function decorated with `@server.tool()`.
   - The **docstring** becomes the tool description shown to the LLM.
   - **Type hints** on parameters are converted to JSON Schema automatically.

> **Security note**: Never give an LLM unrestricted filesystem access in production.
> Always scope operations to an explicit base directory.

In [ ]:
# ── The FastMCP setup pattern (filesystem server) ─────────────────────────
# Every MCP server in this project follows the same three-step recipe.
# Let's inspect the filesystem server to see it in action.

import inspect
from pathlib import Path

# Step 1: Create a FastMCP instance with a name and instructions.
# The "instructions" field tells the LLM what this server can do.
print("=== Step 1: Server instantiation ===")
print(f"  server = FastMCP(")
print(f'      name="{filesystem_server.server.name}",')
print(f'      instructions="{filesystem_server.server.instructions}"')
print(f"  )")
print()

# Step 2: Configure safety boundaries (filesystem-specific).
print("=== Step 2: Safety configuration ===")
print(f"  BASE_DIR = Path.cwd()  # all paths resolved relative to this")
print()

# Step 3: Define tools with @server.tool() — let's look at the
# source code of `read_file` to see the decorator + type hints pattern.
print("=== Step 3: Tool definition (read_file) ===")
print(inspect.getsource(filesystem_server.read_file))

## The Database Server and API Server

AgentExplorr ships two more server archetypes that demonstrate how to wrap different
back-ends behind MCP tools.

### Database Server (`database_server.py`)

Exposes **SQLite** as a set of read-only tools — the foundation for **Text-to-SQL**,
one of the most practical LLM applications.

| Tool | Purpose |
|------|---------|
| `list_tables` | Enumerate every table and its row count |
| `describe_table` | Return column names, types, and constraints |
| `run_query` | Execute a **SELECT-only** SQL query |

Safety is enforced by `_is_read_only()`, which rejects any query containing
`INSERT`, `UPDATE`, `DELETE`, `DROP`, or other mutation keywords, and blocks
multi-statement injection (`SELECT 1; DROP TABLE users`).

### API Server (`api_server.py`)

Wraps the **Open-Meteo** weather REST API (free, no API key) to show the general
pattern for turning any HTTP API into MCP tools.

| Tool | Purpose |
|------|---------|
| `get_current_weather` | Current conditions by city name or lat/lon |
| `get_forecast` | Multi-day forecast (1-7 days) |

The pattern is the same for *any* REST API: map tool parameters to query
parameters, make an HTTP request with `httpx`, and format the JSON response
into a human-readable string for the LLM.

In [ ]:
# ── How different MCP servers expose tools ─────────────────────────────────
# Let's compare the tool signatures across all three servers.
# Each @server.tool() function automatically generates a JSON Schema
# that describes its parameters so the LLM knows how to call it.

import inspect

servers = {
    "Filesystem": filesystem_server,
    "API (Weather)": api_server,
    "Database": database_server,
}

for label, mod in servers.items():
    print(f"{'=' * 60}")
    print(f"  SERVER: {label}  (name={mod.server.name!r})")
    print(f"{'=' * 60}")

    # Collect all @server.tool() functions defined in the module.
    # They are plain module-level functions decorated by the server.
    tool_funcs = [
        (name, obj)
        for name, obj in inspect.getmembers(mod, inspect.isfunction)
        if not name.startswith("_")  # skip private helpers
    ]

    for func_name, func in tool_funcs:
        sig = inspect.signature(func)
        doc_first_line = (func.__doc__ or "").strip().split("\n")[0]
        print(f"\n  @server.tool()")
        print(f"  def {func_name}{sig}")
        print(f"      \"\"\"{doc_first_line}\"\"\"")

    print()

## Key Takeaways

1. **One pattern, many back-ends.** Every MCP server follows the same recipe:
   create a `FastMCP` instance, then decorate plain functions with `@server.tool()`.
   The SDK handles JSON-RPC serialization, transport, and schema generation automatically.

2. **Type hints become schemas.** The parameter type hints on your tool functions are
   converted to JSON Schema so the LLM knows exactly what arguments to pass.

3. **Docstrings become descriptions.** The function docstring is what the LLM reads
   to decide *when* to call a tool. Clear, concise docstrings lead to better tool use.

4. **Security is your responsibility.** MCP itself is transport-agnostic and does not
   enforce access control. Each server must implement its own guardrails:
   - Filesystem: path-traversal protection via `_resolve_safe_path()`
   - Database: read-only query enforcement via `_is_read_only()`
   - API: rate limiting and input validation

5. **Servers run standalone.** Each server can be started with
   `python -m agentexplorr.mcp.servers.<name>` and communicates over stdio.

## Next Steps

- **Notebook 02 (MCP Client Usage)** — learn how a client connects to these servers,
  discovers tools, and invokes them on behalf of an LLM.
- **Try it yourself** — add a new tool to the filesystem server (e.g., `file_info`
  that returns size, modification time, and permissions).
- **Read the spec** — https://spec.modelcontextprotocol.io/ for the full protocol details.